* Data are fetched in reverse order from oldest to newer. So, use reverse indexing to read the data.

In [8]:
from os.path import split

import requests
import pandas as pd
import numpy as np
import json
import xlsxwriter
from xlsxwriter.workbook import Workbook
import csv

url = "https://data.ny.gov/api/views/d6yy-54nr/rows.json?accessType=DOWNLOAD"

r = requests.get(url)
r.raise_for_status()
j = r.json()

# Column names from metadata
cols = [c["name"] for c in j["meta"]["view"]["columns"]]

# Build dataframe
df = pd.DataFrame(j["data"], columns=cols)

# The first columns are Socrata internal fields (like :id, :created_at, :updated_at, etc.)
# Keep only "real" dataset columns by dropping those that start with ':'
real_cols = [c for c in df.columns if not str(c).startswith(":")]
df = df[real_cols]

#print(df.head())#prints oldest 5 draws
#print(df.tail()) #prints most recent 5 draws



In [9]:
#printing dataframe info
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1940 entries, 0 to 1939
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sid              1940 non-null   str   
 1   id               1940 non-null   str   
 2   position         1940 non-null   int64 
 3   created_at       1940 non-null   int64 
 4   created_meta     0 non-null      object
 5   updated_at       1940 non-null   int64 
 6   updated_meta     0 non-null      object
 7   meta             1940 non-null   str   
 8   Draw Date        1940 non-null   str   
 9   Winning Numbers  1940 non-null   str   
 10  Multiplier       1730 non-null   str   
dtypes: int64(3), object(2), str(6)
memory usage: 166.8+ KB


In [10]:
print(df.columns)

Index(['sid', 'id', 'position', 'created_at', 'created_meta', 'updated_at',
       'updated_meta', 'meta', 'Draw Date', 'Winning Numbers', 'Multiplier'],
      dtype='str')


In [11]:
#approx 12 years of history
print(df.shape)
years_of_data = 1940/156
years_of_data, "years of data"

(1940, 11)


(12.435897435897436, 'years of data')

In [12]:
#dropping unwanted columns (noise)
#keeping the wanted ones. I renamed Mult with x

colum_to_keep =["Draw Date","Winning Numbers","x"]
colum_to_drop = ['sid', 'id', 'position', 'created_at', 'created_meta', 'updated_at','updated_meta', 'meta']

In [13]:
df.rename(columns={"Multiplier": "x"}, inplace=True)

In [14]:
len(colum_to_keep)


3

In [15]:
len(colum_to_drop)

8

In [16]:
df[colum_to_keep]

,Draw Date,Winning Numbers,x
0,2020-09-26T00:00:00,11 21 27 36 62 24,3
1,2020-09-30T00:00:00,14 18 36 49 67 18,2
2,2020-10-03T00:00:00,18 31 36 43 47 20,2
3,2020-10-07T00:00:00,06 24 30 53 56 19,2
4,2020-10-10T00:00:00,05 18 23 40 50 18,3
...,...,...,...
1935,2026-05-04T00:00:00,30 36 42 60 63 13,2
1936,2026-05-06T00:00:00,18 27 51 65 68 05,3
1937,2026-05-09T00:00:00,15 41 46 47 56 22,2
1938,2026-05-11T00:00:00,24 30 37 56 64 07,3


In [17]:
#dropping unwanted columns (noise)
df[colum_to_drop]

,sid,id,position,created_at,created_meta,updated_at,updated_meta,meta
0,row-nx2r_usun.g52b,00000000-0000-0000-9D99-B9EA775FDC5E,0,1619710755,None,1619710755,None,{ }
1,row-e8ru_54jr.pm4v,00000000-0000-0000-43B5-52ACB7706E11,0,1619710755,None,1619710755,None,{ }
2,row-tq8m.jys2~s8vu,00000000-0000-0000-A5B2-0AAE86635EA3,0,1619710755,None,1619710755,None,{ }
3,row-fyha_3jj2-8cdk,00000000-0000-0000-8755-E68E26427D31,0,1619710755,None,1619710755,None,{ }
4,row-t7k7-dhjv-h47w,00000000-0000-0000-EF27-CF03FBB38022,0,1619710755,None,1619710755,None,{ }
...,...,...,...,...,...,...,...,...
1935,row-zej2~5dfq-g6zg,00000000-0000-0000-0E97-DBDEECDA4456,0,1777966405,None,1777966405,None,{ }
1936,row-pwwq.27uc~3fid,00000000-0000-0000-5F30-A9869787BB01,0,1778139250,None,1778139250,None,{ }
1937,row-hi9s~52un-v4mz,00000000-0000-0000-3935-44984B742D78,0,1778398432,None,1778398432,None,{ }
1938,row-b8pg-enbi.g8kf,00000000-0000-0000-ED54-5A030A84F1A7,0,1778571665,None,1778571665,None,{ }


In [18]:
#checking duplicates in the dataset. False is none. In this case, it could not find any duplicated drawings in the dataset since 2010.
df.duplicated().value_counts()

False    1940
Name: count, dtype: int64

In [19]:
df[colum_to_keep]

,Draw Date,Winning Numbers,x
0,2020-09-26T00:00:00,11 21 27 36 62 24,3
1,2020-09-30T00:00:00,14 18 36 49 67 18,2
2,2020-10-03T00:00:00,18 31 36 43 47 20,2
3,2020-10-07T00:00:00,06 24 30 53 56 19,2
4,2020-10-10T00:00:00,05 18 23 40 50 18,3
...,...,...,...
1935,2026-05-04T00:00:00,30 36 42 60 63 13,2
1936,2026-05-06T00:00:00,18 27 51 65 68 05,3
1937,2026-05-09T00:00:00,15 41 46 47 56 22,2
1938,2026-05-11T00:00:00,24 30 37 56 64 07,3


In [20]:
Last_5_draws =df[["Draw Date","Winning Numbers"]].tail()
Last_5_draws

,Draw Date,Winning Numbers
1935,2026-05-04T00:00:00,30 36 42 60 63 13
1936,2026-05-06T00:00:00,18 27 51 65 68 05
1937,2026-05-09T00:00:00,15 41 46 47 56 22
1938,2026-05-11T00:00:00,24 30 37 56 64 07
1939,2026-05-13T00:00:00,22 31 52 56 67 15


In [21]:
Cleaned_Drawings = df[colum_to_keep]


In [22]:
Cleaned_Drawings

,Draw Date,Winning Numbers,x
0,2020-09-26T00:00:00,11 21 27 36 62 24,3
1,2020-09-30T00:00:00,14 18 36 49 67 18,2
2,2020-10-03T00:00:00,18 31 36 43 47 20,2
3,2020-10-07T00:00:00,06 24 30 53 56 19,2
4,2020-10-10T00:00:00,05 18 23 40 50 18,3
...,...,...,...
1935,2026-05-04T00:00:00,30 36 42 60 63 13,2
1936,2026-05-06T00:00:00,18 27 51 65 68 05,3
1937,2026-05-09T00:00:00,15 41 46 47 56 22,2
1938,2026-05-11T00:00:00,24 30 37 56 64 07,3


In [23]:
#printing date and winning numbers plus multiplier
df[["Draw Date","Winning Numbers","x"]]

,Draw Date,Winning Numbers,x
0,2020-09-26T00:00:00,11 21 27 36 62 24,3
1,2020-09-30T00:00:00,14 18 36 49 67 18,2
2,2020-10-03T00:00:00,18 31 36 43 47 20,2
3,2020-10-07T00:00:00,06 24 30 53 56 19,2
4,2020-10-10T00:00:00,05 18 23 40 50 18,3
...,...,...,...
1935,2026-05-04T00:00:00,30 36 42 60 63 13,2
1936,2026-05-06T00:00:00,18 27 51 65 68 05,3
1937,2026-05-09T00:00:00,15 41 46 47 56 22,2
1938,2026-05-11T00:00:00,24 30 37 56 64 07,3


In [24]:
#printing the last 50 draws. This Database prints values from older to newer. In order words in reverse order
df[["Draw Date","Winning Numbers","x"]][-50:].sort_values(by="Draw Date", ascending=False)


,Draw Date,Winning Numbers,x
1939,2026-05-13T00:00:00,22 31 52 56 67 15,2
1938,2026-05-11T00:00:00,24 30 37 56 64 07,3
1937,2026-05-09T00:00:00,15 41 46 47 56 22,2
1936,2026-05-06T00:00:00,18 27 51 65 68 05,3
1935,2026-05-04T00:00:00,30 36 42 60 63 13,2
1934,2026-05-02T00:00:00,25 37 42 52 65 14,3
1933,2026-04-29T00:00:00,03 19 35 51 67 15,2
1932,2026-04-27T00:00:00,18 31 33 36 62 03,3
1931,2026-04-25T00:00:00,04 30 36 52 57 02,3
1930,2026-04-22T00:00:00,24 29 32 49 63 11,2


In [25]:
last_year = df[["Winning Numbers"]][-150:]
for x in last_year:
    if ((5, 28, 34, 37, 55, 17)) in last_year:
        print("yeah", last_year)
    else:
        print("no")
        break


no


In [26]:
#df.to_excel("Drawings.xlsx")

In [27]:
#pd.read_excel("drawings.xlsx")

In [28]:
#save as a csv file
df.to_csv("drawings.csv")

In [29]:
pd.read_csv("drawings.csv")

,Unnamed: 0,sid,id,position,created_at,created_meta,updated_at,updated_meta,meta,Draw Date,Winning Numbers,x
0,0,row-nx2r_usun.g52b,00000000-0000-0000-9D99-B9EA775FDC5E,0,1619710755,NaN,1619710755,NaN,{ },2020-09-26T00:00:00,11 21 27 36 62 24,3.0
1,1,row-e8ru_54jr.pm4v,00000000-0000-0000-43B5-52ACB7706E11,0,1619710755,NaN,1619710755,NaN,{ },2020-09-30T00:00:00,14 18 36 49 67 18,2.0
2,2,row-tq8m.jys2~s8vu,00000000-0000-0000-A5B2-0AAE86635EA3,0,1619710755,NaN,1619710755,NaN,{ },2020-10-03T00:00:00,18 31 36 43 47 20,2.0
3,3,row-fyha_3jj2-8cdk,00000000-0000-0000-8755-E68E26427D31,0,1619710755,NaN,1619710755,NaN,{ },2020-10-07T00:00:00,06 24 30 53 56 19,2.0
4,4,row-t7k7-dhjv-h47w,00000000-0000-0000-EF27-CF03FBB38022,0,1619710755,NaN,1619710755,NaN,{ },2020-10-10T00:00:00,05 18 23 40 50 18,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1935,1935,row-zej2~5dfq-g6zg,00000000-0000-0000-0E97-DBDEECDA4456,0,1777966405,NaN,1777966405,NaN,{ },2026-05-04T00:00:00,30 36 42 60 63 13,2.0
1936,1936,row-pwwq.27uc~3fid,00000000-0000-0000-5F30-A9869787BB01,0,1778139250,NaN,1778139250,NaN,{ },2026-05-06T00:00:00,18 27 51 65 68 05,3.0
1937,1937,row-hi9s~52un-v4mz,00000000-0000-0000-3935-44984B742D78,0,1778398432,NaN,1778398432,NaN,{ },2026-05-09T00:00:00,15 41 46 47 56 22,2.0
1938,1938,row-b8pg-enbi.g8kf,00000000-0000-0000-ED54-5A030A84F1A7,0,1778571665,NaN,1778571665,NaN,{ },2026-05-11T00:00:00,24 30 37 56 64 07,3.0


In [30]:
type(df["Winning Numbers"])# column

pandas.Series

In [31]:
df["Winning Numbers"][:5]# column

0    11 21 27 36 62 24
1    14 18 36 49 67 18
2    18 31 36 43 47 20
3    06 24 30 53 56 19
4    05 18 23 40 50 18
Name: Winning Numbers, dtype: str

In [32]:
s = df["Winning Numbers"]# column
s[:1]

0    11 21 27 36 62 24
Name: Winning Numbers, dtype: str

In [33]:
nums_all_5 = df["Winning Numbers"]

nums_all_5 = s.apply(lambda x: x[:14])
nums_all_5

0       11 21 27 36 62
1       14 18 36 49 67
2       18 31 36 43 47
3       06 24 30 53 56
4       05 18 23 40 50
             ...      
1935    30 36 42 60 63
1936    18 27 51 65 68
1937    15 41 46 47 56
1938    24 30 37 56 64
1939    22 31 52 56 67
Name: Winning Numbers, Length: 1940, dtype: str

In [34]:
#select the first 5 numbers White balls and convert them to int
s_first5 = s.apply(lambda x: tuple(map(int, x.split()[:5])))
s_first5

0       (11, 21, 27, 36, 62)
1       (14, 18, 36, 49, 67)
2       (18, 31, 36, 43, 47)
3        (6, 24, 30, 53, 56)
4        (5, 18, 23, 40, 50)
                ...         
1935    (30, 36, 42, 60, 63)
1936    (18, 27, 51, 65, 68)
1937    (15, 41, 46, 47, 56)
1938    (24, 30, 37, 56, 64)
1939    (22, 31, 52, 56, 67)
Name: Winning Numbers, Length: 1940, dtype: object

In [35]:
PowerBall = s.apply(lambda x: int(x.split()[5]))
PowerBall

0       24
1       18
2       20
3       19
4       18
        ..
1935    13
1936     5
1937    22
1938     7
1939    15
Name: Winning Numbers, Length: 1940, dtype: int64

In [48]:
#describes the range index
PowerBall[-10:]

1930    11
1931     2
1932     3
1933    15
1934    14
1935    13
1936     5
1937    22
1938     7
1939    15
Name: Winning Numbers, dtype: int64

In [37]:
PowerBall.value_counts()


Winning Numbers
24    82
14    77
4     77
18    76
20    76
5     76
1     75
25    75
21    70
19    67
23    67
26    67
6     67
2     67
13    65
11    65
3     65
9     65
7     64
15    64
10    61
8     61
17    61
12    61
22    59
16    57
29    27
33    22
27    19
32    16
35    15
30    15
28    14
34    13
31    11
36     7
39     6
38     5
37     3
Name: count, dtype: int64

In [38]:
PowerBall.value_counts().to_excel(
    "PowerBall.xlsx", )

In [39]:
pd.read_excel("PowerBall.xlsx")

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

In [ ]:
nums = [1,2,3,4,5,6]
previous = i-1
for i in nums:
    if i == previous:
        print("yeah", i)
    else:
        print("sad")
